# geosite.dat для российских маршрутов

Собирает `geosite.dat` с двумя списками:

| Список | Содержимое | Маршрут |
|:---|:---|:---|
| `ru-blocked` | домены, заблокированные в России | через прокси |
| `ru-forbidden` | домены, которые сами закрывают доступ российским адресам | через прокси |

Обе ветки проходят разную обработку. `ru-blocked` собирается из реестра
Роскомнадзора, куда по закону попадают решения не только о цензуре, но и о
казино, фишинге, наркотиках и пиратстве — этот объём нужно отфильтровать, иначе
итоговый файл не уложится в память мобильного клиента на iOS. `ru-forbidden`
курируется вручную сообществом и в чистке не нуждается, но проверяется на
российские домены, попавшие туда по ошибке.

## Ограничение по памяти

Расширение `NEPacketTunnelProvider` на iOS имеет лимит 50 МБ на процесс
целиком, включая сетевой стек и буферы пакетов. На данные списка остаётся
25–35 МБ. Проверка размера — последняя ячейка.

## Формат правил

Все домены пишутся как правило типа `Domain` (значение 2 в protobuf-схеме).
Оно матчит и сам домен, и все его поддомены: запись `example.com` покрывает
`api.example.com` и `a.b.example.com`. Перечислять поддомены отдельно не нужно,
такие записи удаляются на этапе дедупликации.

## Подключение в xray

```json
"rules": [
  { "type": "field", "domain": ["geosite:ru-blocked", "geosite:ru-forbidden"],
    "outboundTag": "proxy" },
  { "type": "field", "network": "tcp,udp", "outboundTag": "direct" }
]
```

In [1]:
!pip install -q requests dnspython scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 6.7 MB/s eta 0:00:00


In [2]:
import bisect
import concurrent.futures as cf
import gzip
import ipaddress
import os
import random
import re
import time

import dns.resolver
import numpy as np
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split

## 1. Источники

In [3]:
# antizapret и isitblockedinrussia отдельными источниками не подключены: оба
# читают реестр из zapret-info/z-i, который скачивается ниже напрямую.
RKN_SOURCES = {
    "antifilter_community": "https://community.antifilter.download/list/domains.lst",
    "re_filter": "https://raw.githubusercontent.com/1andrevich/Re-filter-lists/refs/heads/main/domains_all.lst",
}

GEOBLOCK_SOURCES = {
    "dartraiden": "https://raw.githubusercontent.com/dartraiden/no-russia-hosts/refs/heads/master/hosts.txt",
    "itdoginfo": "https://raw.githubusercontent.com/itdoginfo/allow-domains/refs/heads/main/Categories/geoblock.lst",
    "internet_helper": "https://raw.githubusercontent.com/Internet-Helper/Unblock-for-Russia/refs/heads/main/geoblock.lst",
}

In [5]:
ZAPRET_DUMP_URL = "https://raw.githubusercontent.com/zapret-info/z-i/master/dump.csv.gz"
VERNETTE_API_URL = "https://api.github.com/repos/vernette/rulesets/contents/raw"
VERNETTE_RAW_BASE = "https://raw.githubusercontent.com/vernette/rulesets/master/raw/"
ADGUARD_PHISHING_URL = "https://raw.githubusercontent.com/AdguardTeam/HostlistsRegistry/main/filters/security/filter_30_PhishingURLBlocklist/filter.txt"

HEADERS = {"User-Agent": "geosite-ru-builder"}
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

In [6]:
def fetch_plain_list(url, timeout=120):
    """Читает список вида «один домен на строку»."""
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return [line.strip().lower() for line in r.text.splitlines()
            if line.strip() and not line.startswith(("#", "!"))]

In [7]:
HOSTS_LINE = re.compile(r"^0\.0\.0\.0\s+([a-z0-9.\-]+)$")


def fetch_hosts_file(url, timeout=180):
    """Читает hosts-формат: «0.0.0.0 domain»."""
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    out = []
    for line in r.text.splitlines():
        m = HOSTS_LINE.match(line.strip().lower())
        if m and m.group(1) != "0.0.0.0":
            out.append(m.group(1))
    return out

In [8]:
def fetch_zapret_dump(url=ZAPRET_DUMP_URL, timeout=300):
    """Реестр Роскомнадзора из zapret-info/z-i.

    Файл доступен только в сжатом виде: dump.csv без .gz отдаёт 404.
    Строка имеет три поля через ';': список IP, домен, список URL.
    Поле с органом и номером решения, описанное в старых схемах реестра,
    в текущем дампе отсутствует — категорию блокировки из него получить
    нельзя, поэтому разделение на мусор и цензуру идёт по самому домену
    (раздел 4).
    Кодировка cp1251. Около 40% доменов записаны с префиксом '*.', он
    снимается: правило типа Domain и так покрывает поддомены.
    """
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    text = gzip.decompress(r.content).decode("cp1251", errors="replace")

    domains = []
    for line in text.splitlines():
        if line.startswith("Updated:"):
            continue
        parts = line.split(";")
        if len(parts) < 2:
            continue
        domain = parts[1].strip().lower()
        if domain.startswith("*."):
            domain = domain[2:]
        if domain:
            domains.append(domain)
    return domains

In [10]:
def fetch_vernette_rulesets(timeout=120):
    """Все .txt из vernette/rulesets/raw.

    Список файлов запрашивается через API, а не хардкодится: набор сервисов
    в репозитории меняется. Берутся и сводный файл, и отдельные сервисные —
    сводный не является надмножеством, часть сервисов из него исключена
    автором намеренно.
    """
    resp = requests.get(VERNETTE_API_URL, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    entries = resp.json()
    if isinstance(entries, dict):
        raise RuntimeError(entries.get("message", "unexpected API response"))

    domains = set()
    for name in (e["name"] for e in entries if e["name"].endswith(".txt") and e["name"] != "rkn.txt"):
        try:
            domains.update(fetch_plain_list(VERNETTE_RAW_BASE + name))
        except Exception as exc:
            print(f"  пропущен {name}: {exc}")
        time.sleep(0.2)
    return sorted(domains)

In [11]:
rkn_raw = []
for name, url in RKN_SOURCES.items():
    try:
        domains = fetch_plain_list(url)
        rkn_raw.extend(domains)
        print(f"{name}: {len(domains):,}")
    except Exception as exc:
        print(f"{name}: недоступен — {exc}")

antifilter_community: 486
re_filter: 81,061


In [12]:
zapret_domains = fetch_zapret_dump()
rkn_raw.extend(zapret_domains)
print(f"zapret-info/z-i: {len(zapret_domains):,}")

zapret-info/z-i: 1,202,380


In [13]:
geoblock_raw = []
for name, url in GEOBLOCK_SOURCES.items():
    try:
        domains = fetch_plain_list(url)
        geoblock_raw.extend(domains)
        print(f"{name}: {len(domains):,}")
    except Exception as exc:
        print(f"{name}: недоступен — {exc}")

dartraiden: 729
itdoginfo: 466
internet_helper: 962


In [14]:
vernette_domains = fetch_vernette_rulesets()
geoblock_raw.extend(vernette_domains)
print(f"vernette/rulesets: {len(vernette_domains):,}")

vernette/rulesets: 798


In [15]:
print(f"ru-blocked, сырых записей:   {len(rkn_raw):,}")
print(f"ru-forbidden, сырых записей: {len(geoblock_raw):,}")

ru-blocked, сырых записей:   1,283,927
ru-forbidden, сырых записей: 2,955


## 2. Дедупликация

Два прохода. Первый снимает точные повторы после нормализации регистра,
завершающей точки и префикса `www.`. Второй убирает записи, уже покрытые
правилом родительского домена: при наличии `example.com` записи
`sub.example.com` и `a.b.example.com` избыточны, так как правило типа `Domain`
матчит поддомены любой глубины. Построчное сравнение такие пары не находит —
строки не совпадают.

In [16]:
def dedup_exact(domains):
    seen, out = set(), []
    for domain in domains:
        domain = domain.strip().lower().rstrip(".")
        if domain.startswith("www."):
            domain = domain[4:]
        if domain and domain not in seen:
            seen.add(domain)
            out.append(domain)
    return out

In [17]:
def dedup_by_coverage(domains):
    """Удаляет домены, покрытые правилом уже принятого родителя.

    Сортировка по числу меток гарантирует, что родитель рассматривается
    раньше потомка, поэтому достаточно одного прохода.
    """
    kept, kept_set = [], set()
    for domain in sorted(set(domains), key=lambda d: d.count(".")):
        labels = domain.split(".")
        if any(".".join(labels[i:]) in kept_set for i in range(1, len(labels))):
            continue
        kept.append(domain)
        kept_set.add(domain)
    return kept

In [18]:
def dedup(domains):
    exact = dedup_exact(domains)
    covered = dedup_by_coverage(exact)
    stats = {
        "на входе": len(domains),
        "точных повторов": len(domains) - len(exact),
        "покрыто родителем": len(exact) - len(covered),
        "на выходе": len(covered),
    }
    return covered, stats

In [19]:
rkn, rkn_stats = dedup(rkn_raw)
geoblock, geoblock_stats = dedup(geoblock_raw)
print("ru-blocked:  ", rkn_stats)
print("ru-forbidden:", geoblock_stats)

ru-blocked:   {'на входе': 1283927, 'точных повторов': 291126, 'покрыто родителем': 112311, 'на выходе': 880490}
ru-forbidden: {'на входе': 2955, 'точных повторов': 1701, 'покрыто родителем': 108, 'на выходе': 1146}


## 3. Российские домены в ветке `ru-forbidden`

Домен, физически размещённый в России, не может блокировать российских
пользователей — попав в этот список, он будет отправлен через зарубежный
прокси.

Проверяются три сигнала: доменная зона, страна A-записи и страны NS-серверов.
Список известных российских хостеров не ведётся — вместо него IP-адреса
NS-серверов и самого домена сопоставляются с офлайн-базой диапазонов. Так
детектор не требует правки при появлении нового хостера: `habr.com` использует
собственные NS `habradns.net`, которые не совпали бы ни с одним маркером, но
его A-запись определяется как RU.

База: `sapics/ip-location-db`, файл `server-country-ipv4.csv`, лицензия PDDL,
без ключей и лимитов на запросы.

In [20]:
GEOIP_URL = "https://github.com/sapics/ip-location-db/releases/download/latest/server-country-ipv4.csv"
RU_TLDS = {"ru", "su", "xn--p1ai"}  # xn--p1ai — punycode для .рф

In [21]:
class CountryDB:
    """Диапазоны IPv4 → код страны, поиск двоичным делением."""

    def __init__(self, csv_text):
        self.starts, self.ends, self.codes = [], [], []
        for line in csv_text.splitlines():
            start, end, code = line.split(",")
            self.starts.append(int(ipaddress.IPv4Address(start)))
            self.ends.append(int(ipaddress.IPv4Address(end)))
            self.codes.append(code)

    def lookup(self, ip):
        try:
            value = int(ipaddress.IPv4Address(ip))
        except ipaddress.AddressValueError:
            return None
        idx = bisect.bisect_right(self.starts, value) - 1
        if idx >= 0 and value <= self.ends[idx]:
            return self.codes[idx]
        return None

In [22]:
geoip = CountryDB(requests.get(GEOIP_URL, headers=HEADERS, timeout=300).text)
print(f"диапазонов в базе: {len(geoip.starts):,}")

диапазонов в базе: 273,517


In [23]:
def resolve(name, rdtype, timeout=5.0):
    try:
        answers = dns.resolver.resolve(name, rdtype, lifetime=timeout)
    except Exception:
        return []
    if rdtype == "NS":
        return [str(r.target).rstrip(".").lower() for r in answers]
    return [str(r) for r in answers]

In [24]:
def ru_signals(domain):
    """Возвращает список сработавших признаков российской принадлежности."""
    signals = []

    if domain.rsplit(".", 1)[-1] in RU_TLDS:
        signals.append("tld")

    for ip in resolve(domain, "A")[:2]:
        if geoip.lookup(ip) == "RU":
            signals.append(f"a={ip}")
            break

    ns_countries = []
    for ns in resolve(domain, "NS")[:3]:
        ns_ips = resolve(ns, "A")[:1]
        if ns_ips:
            ns_countries.append(geoip.lookup(ns_ips[0]))
    if ns_countries and all(cc == "RU" for cc in ns_countries):
        signals.append("ns")

    return signals

In [25]:
def split_ru_domains(domains, workers=24):
    clean, flagged = [], []
    with cf.ThreadPoolExecutor(max_workers=workers) as pool:
        for domain, signals in zip(domains, pool.map(ru_signals, domains)):
            (flagged if signals else clean).append((domain, signals))
    return [d for d, _ in clean], flagged

In [26]:
geoblock_clean, ru_flagged = split_ru_domains(geoblock)
print(f"оставлено: {len(geoblock_clean):,}")
print(f"помечено как российские: {len(ru_flagged):,}")
for domain, signals in ru_flagged[:40]:
    print(f"  {domain:40s} {','.join(signals)}")

оставлено: 1,140
помечено как российские: 6
  4pda.ru                                  tld,ns
  4pda.ws                                  ns
  kemono.su                                tld
  habr.com                                 a=178.248.237.68
  intel.ru                                 tld
  gpt3-openai.com                          a=5.101.152.54,ns


## 4. Отделение цензуры от прочих блокировок

Реестр не содержит поля с причиной блокировки, поэтому категория определяется
по самому домену. Работают два слоя.

**Слой 1 — сверка с категоризованными фидами.** UT1 (Université Toulouse
Capitole, зеркало обновляется раз в сутки, CC BY-SA) раскладывает домены по 67
категориям; StevenBlack даёт отдельный список порнографии. Слой точный, но
запаздывающий: замер на 81 059 доменах Re-filter показал пересечение с UT1
(phishing + malware + gambling) на уровне 0.70%. Пополнение UT1 — 50–300
записей в день вручную, а мусорные домены реестра размножаются быстрее.
Как единственный механизм слой непригоден.

Категории `phishing` и `malware` в UT1 совпадают на 98.1% и считаются одним
источником. Категории `vpn`, `redirector`, `doh`, `shortener` в мусор **не**
включаются: для школьного прокси, под который UT1 создавался, это нежелательный
трафик, для этого списка — целевой.

**Слой 2 — классификатор на символьных n-граммах.** Обучается на разметке UT1:
положительный класс — gambling, drogue, cryptojacking, ddos, stalkerware;
отрицательный — shopping, bank, press, jobsearch, games. Обе стороны из одного
источника, поэтому модель учится на различии категорий, а не на артефактах
разных сборщиков. Категория `games` в отрицательном классе обязательна: без неё
модель принимает игровые студии за гемблинг (`platinumgames.org` получал 0.998).

Выбор архитектуры. В задачах детекции вредоносных URL трансформеры дают лучший
результат — URLTran при FPR 0.01% достигает TPR 86.8% против 71.2% у char-CNN
URLNet. Но там решается другая задача: бинарная детекция свежего фишинга при
экстремально низком FPR, где выигрывает понимание контекста. Здесь разделяются
тематические категории, и сигнал буквально лексический — бренд-подстроки
(`casino`, `vulkan`, `1x`, `bet`) и их искажения, на которых n-граммы работают
вровень. Модель переобучается при каждом обновлении фидов, а раннеры GitHub
Actions не имеют GPU: 10 секунд на CPU против файнтюна трансформера — решающий
аргумент при равном качестве.

Порог не выбирается вручную: берутся два значения по кривой precision-recall.
Всё выше порога precision ≥ 0.99 удаляется, интервал до порога precision ≥ 0.95
идёт в отдельную корзину на просмотр, остальное остаётся.

Ошибки, которые остаются на этих порогах: болгарское издание `mediapool.bg`
получает 0.999 из-за подстроки `pool`, характерной для майнинговых пулов.
Модель работает с именем домена и не может отличить такой случай без обращения
к содержимому сайта. Контрольная ячейка ниже печатает верх удаляемой выборки —
её нужно просматривать при изменении состава источников.

In [27]:
UT1_BASE = "https://raw.githubusercontent.com/olbat/ut1-blacklists/master/blacklists/{}/domains"

# phishing не указан: совпадает с malware на 98.1%
UT1_JUNK = ["gambling", "drogue", "cryptojacking", "ddos", "stalkerware", "malware"]
UT1_LEGIT = ["shopping", "bank", "press", "jobsearch", "games"]

# Категория adult недоступна через raw.githubusercontent: файл превышает лимит
# размера GitHub. Порнография берётся из расширений StevenBlack.
PORN_URLS = [
    "https://raw.githubusercontent.com/StevenBlack/hosts/master/extensions/porn/sinfonietta/hosts",
    "https://raw.githubusercontent.com/StevenBlack/hosts/master/extensions/porn/clefspeare13/hosts",
]

In [28]:
def fetch_ut1(category, timeout=180):
    return set(fetch_plain_list(UT1_BASE.format(category), timeout=timeout))

In [29]:
ut1_junk = {}
for category in UT1_JUNK:
    ut1_junk[category] = fetch_ut1(category)
    print(f"{category}: {len(ut1_junk[category]):,}")

gambling: 32,247
drogue: 603
cryptojacking: 11,491
ddos: 421
stalkerware: 525
malware: 252,120


In [30]:
porn_domains = set()
for url in PORN_URLS:
    porn_domains.update(fetch_hosts_file(url))
print(f"porn: {len(porn_domains):,}")

porn: 74,472


In [31]:
class DomainMatcher:
    """Проверка вхождения домена в набор с учётом родительских доменов.

    Запись example.com в наборе считается покрывающей sub.example.com:
    поддомен фишингового хостинга остаётся фишинговым.
    """

    def __init__(self, domains):
        self.domains = set(domains)

    def match(self, domain):
        if domain in self.domains:
            return True
        labels = domain.split(".")
        return any(".".join(labels[i:]) in self.domains for i in range(1, len(labels)))

In [32]:
feed_matcher = DomainMatcher(set().union(*ut1_junk.values()) | porn_domains)

by_feeds, remaining = [], []
for domain in rkn:
    (by_feeds if feed_matcher.match(domain) else remaining).append(domain)

print(f"отсеяно по фидам: {len(by_feeds):,} ({len(by_feeds) / len(rkn):.2%})")
print(f"осталось:         {len(remaining):,}")

отсеяно по фидам: 22,284 (2.53%)
осталось:         858,206


In [33]:
ut1_legit = set()
for category in UT1_LEGIT:
    ut1_legit |= fetch_ut1(category)

train_junk = sorted(set().union(*(ut1_junk[c] for c in
                                  ["gambling", "drogue", "cryptojacking", "ddos", "stalkerware"]))
                    - ut1_legit)
train_legit = sorted(ut1_legit)
print(f"обучающая выборка: мусор {len(train_junk):,}, легитимные {len(train_legit):,}")

обучающая выборка: мусор 45,034, легитимные 142,666


In [34]:
X = train_junk + train_legit
y = np.array([1] * len(train_junk) + [0] * len(train_legit))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# char_wb ограничивает n-граммы границами меток домена: подстрока не
# «перетекает» через точку и остаётся признаком конкретной метки.
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5),
                             min_df=3, sublinear_tf=True, max_features=300_000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=3000, C=10, class_weight="balanced")
model.fit(X_train_vec, y_train)
print(f"признаков: {X_train_vec.shape[1]:,}")

признаков: 206,516


In [35]:
scores = model.predict_proba(X_test_vec)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, scores):.4f}")
print(classification_report(y_test, scores > 0.5,
                            target_names=["легитимные", "мусор"], digits=3))

ROC-AUC: 0.9799
              precision    recall  f1-score   support

  легитимные      0.973     0.963     0.968     35667
       мусор      0.886     0.916     0.901     11258

    accuracy                          0.952     46925
   macro avg      0.930     0.939     0.934     46925
weighted avg      0.952     0.952     0.952     46925



In [40]:
from scipy.stats import gaussian_kde

remaining_scores = model.predict_proba(vectorizer.transform(remaining))[:, 1]

# Пороги по кривой precision-recall откалиброваны на UT1, где доля мусора 24%.
# В остатке реестра она принципиально выше, а precision зависит от доли классов,
# поэтому перенесённый порог оказывается завышенным. Здесь порог берётся из
# самих целевых оценок: их распределение двумодально (легитимные у нуля,
# казино у единицы), минимум плотности между модами и есть точка разделения.
kde = gaussian_kde(remaining_scores, bw_method=0.05)
grid = np.linspace(0.30, 0.95, 400)
DROP_THRESHOLD = float(grid[np.argmin(kde(grid))])
print(f"порог по впадине: {DROP_THRESHOLD:.3f}")

порог по впадине: 0.728


In [46]:
PSL_URL = "https://raw.githubusercontent.com/publicsuffix/list/master/public_suffix_list.dat"
TOP_URL = "https://raw.githubusercontent.com/zer0h/top-1000000-domains/master/top-1000000-domains"

psl_text = requests.get(PSL_URL, headers=HEADERS, timeout=120).text
PUBLIC_SUFFIXES = {line.strip() for line in psl_text.splitlines()
                   if line.strip() and not line.startswith("//")}
TOP_DOMAINS = set(fetch_plain_list(TOP_URL))


def registrable(domain):
    """Домен вместе с одной меткой сверх публичного суффикса."""
    labels = domain.split(".")
    for i in range(1, len(labels)):
        if ".".join(labels[i:]) in PUBLIC_SUFFIXES:
            return ".".join(labels[i - 1:])
    return domain


def protected(domain):
    """Причина, по которой домен не удаляется по решению модели."""
    if domain in PUBLIC_SUFFIXES:
        return "public-suffix"
    reg = registrable(domain)
    if len(reg.split(".")[0]) < 4:
        return "short-label"
    if reg in TOP_DOMAINS:
        return "top-1m"
    return None

In [47]:
by_model, kept, shielded = [], [], []
for domain, score in zip(remaining, remaining_scores):
    if score <= DROP_THRESHOLD:
        kept.append(domain)
    elif (reason := protected(domain)):
        shielded.append((domain, score, reason))
        kept.append(domain)
    else:
        by_model.append(domain)

print(f"удалено моделью: {len(by_model):,}, защищено: {len(shielded):,}")
for domain, score, reason in shielded[:30]:
    print(f"  {score:.3f}  {reason:14s} {domain}")

удалено моделью: 334,156, защищено: 3,607
  0.961  short-label    n
  1.000  short-label    top.casino
  0.775  short-label    upx.vodka
  0.998  top-1m         betfaircasino.com
  0.999  top-1m         myjackpotcasino.com
  0.812  top-1m         vkadre.ws
  1.000  top-1m         1xbet55.com
  0.997  top-1m         optibet.ee
  0.880  top-1m         xxx-ok.com
  0.979  top-1m         play2wincasinos.com
  0.876  top-1m         tushkan.tv
  0.998  top-1m         casinohuone.com
  0.902  top-1m         kittybingo.com
  0.995  top-1m         optibet.com
  0.976  top-1m         lider-bet.com
  0.771  top-1m         palaceofchance.com
  0.952  top-1m         surebet247.com
  0.811  short-label    npb.finance
  0.814  short-label    1go.mom
  0.894  top-1m         twimg.com
  0.854  top-1m         lotto-sh.de
  0.776  top-1m         tvnet.lv
  0.979  top-1m         777-slot.com
  0.953  short-label    mg2.at
  0.980  top-1m         casinophonebill.com
  0.754  top-1m         azino777.com
  0

In [44]:
# Контрольная выборка: домены с высшим баллом должны быть узнаваемым мусором,
# а корзина просмотра — содержать спорные случаи, а не очевидную цензуру.
print("удалено моделью, топ-20:")
for domain in sorted(zip(remaining_scores, remaining), reverse=True)[:20]:
    print(f"  {domain[0]:.3f}  {domain[1]}")

удалено моделью, топ-20:
  1.000  vulkancasino.bid
  1.000  joycasino-slots.bid
  1.000  casino1xbet.bid
  1.000  vulk24casino.bid
  1.000  vulkcasino.bid
  1.000  vulkanbet.bid
  1.000  pokerdom-casino.bid
  1.000  vulkan24-casino.bid
  1.000  vulkancasino-slots.com
  1.000  vulkano-casino-slots.com
  1.000  casino-slotv.bid
  1.000  casino-1xbet.bid
  1.000  vulkan-casino-slots.ru
  1.000  vulkancasino-slots.xyz
  1.000  volnacasino.bid
  1.000  vulcan-casino.bid
  1.000  r7-casino.bid
  1.000  vulkanrussia-casino.bid
  1.000  vulkan-vegas.bid
  1.000  casino-maxslots.bid


In [49]:
# Корзина просмотра по умолчанию остаётся в списке: ложное удаление здесь
# дороже лишней записи. Чтобы удалять её тоже, заменить на kept.
rkn_filtered = kept
print(f"после слоёв 1 и 2: {len(rkn_filtered):,} из {len(rkn):,}")

после слоёв 1 и 2: 520,443 из 880,490


## 5. Список фишинга AdGuard

Отдельный источник поверх UT1: обновляется чаще и покрывает домены, до которых
ручная курация UT1 не дошла.

На текущих данных слой почти холостой — пересечение списка AdGuard (36 664
домена) с Re-filter равно нулю, а с категорией `malware` UT1 составляет 1 831
домен. Популяции разные: AdGuard ведёт свежие фишинговые поддомены на
конструкторах сайтов, реестр — другой класс ресурсов. Слой оставлен как
страховка стоимостью в один HTTP-запрос, содержимое обоих списков меняется
ежедневно.

In [50]:
ADBLOCK_RULE = re.compile(r"^\|\|([a-z0-9.\-]+)\^")


def fetch_adguard_phishing(timeout=120):
    r = requests.get(ADGUARD_PHISHING_URL, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return {m.group(1) for m in
            (ADBLOCK_RULE.match(line.strip()) for line in r.text.splitlines()) if m}

In [51]:
adguard_matcher = DomainMatcher(fetch_adguard_phishing())
before = len(rkn_filtered)
rkn_filtered = [d for d in rkn_filtered if not adguard_matcher.match(d)]
print(f"отсеяно AdGuard: {before - len(rkn_filtered):,}")
print(f"осталось:        {len(rkn_filtered):,}")

отсеяно AdGuard: 2
осталось:        520,441


## 6. Google Safe Browsing

Шаг необязательный и выполняется только при заданном ключе. К этому моменту
список сокращён предыдущими слоями, поэтому квота API расходуется экономно.
Lookup API принимает до 500 URL на запрос.

In [53]:
SAFE_BROWSING_URL = "https://safebrowsing.googleapis.com/v4/threatMatches:find"

try:
    from google.colab import userdata
    SAFE_BROWSING_KEY = userdata.get("GOOGLE_SAFE_BROWSING_KEY")
except Exception:
    SAFE_BROWSING_KEY = os.environ.get("GOOGLE_SAFE_BROWSING_KEY")

In [54]:
def safe_browsing_flags(domains, api_key, batch_size=500):
    flagged = set()
    for start in range(0, len(domains), batch_size):
        batch = domains[start:start + batch_size]
        body = {
            "client": {"clientId": "geosite-ru-builder", "clientVersion": "1.0"},
            "threatInfo": {
                "threatTypes": ["SOCIAL_ENGINEERING", "MALWARE", "UNWANTED_SOFTWARE"],
                "platformTypes": ["ANY_PLATFORM"],
                "threatEntryTypes": ["URL"],
                "threatEntries": [{"url": f"http://{d}"} for d in batch],
            },
        }
        r = requests.post(SAFE_BROWSING_URL, params={"key": api_key}, json=body, timeout=60)
        r.raise_for_status()
        for match in r.json().get("matches", []):
            flagged.add(match["threat"]["url"].split("//", 1)[-1].split("/", 1)[0])
        time.sleep(1)
    return flagged

In [39]:
manual_safe_browsing_run = {
 '100.viromin.com',
 '1wchpa.xyz',
 '1weplj.top',
 '1win-official-zerkalo14.buzz',
 '1win-vrw.top',
 '1winpo2o.top',
 '1wint1.com',
 '1winzzzoff26.top',
 '1wzlow.top',
 '26years.npbpromoru.nl',
 '34kra.com',
 '43.156.75.220',
 '4liveacademia.com.br',
 '6762.info',
 '7.fakaza.cyou',
 '888-starz.kz',
 'a-bizcenter.com',
 'aarescapital.net',
 'aarescapitalltd.net',
 'actavanusfxdrive.com',
 'agamangadi.com',
 'agrofarmtech.com',
 'ajmanivban.com',
 'aldigora.ru',
 'allfinn.ru',
 'alphaultramkts.com',
 'amazingnetscrypto.com',
 'amazonkaseeds.com',
 'amichocolates.com',
 'andover-trades.com',
 'animegoo.top',
 'apexex-change.com',
 'apexoptionfx.com',
 'apexprimevest.com',
 'apextradeshub.net',
 'app.city2000.cc',
 'app.phic4.top',
 'arrowtradify.info',
 'ascensionmountsterling.org',
 'aslltd.cc',
 'assetaccrue.com',
 'assetsmarket.click',
 'astroprofinance.com',
 'astrosvest.com',
 'atlascapital-management.com',
 'atriastudio.ru',
 'aurora-wealth-ai-app.com',
 'auto-tradinginternational.com',
 'avito.id-39.icu',
 'avtoritetkzn.ru',
 'axisledger.live',
 'azbit29f.com',
 'azino777mobile.top',
 'azinobonus777.top',
 'bairbie.ru',
 'basetrade.world',
 'batencoincraft.com',
 'bazaarfx.net',
 'bevoout.life',
 'bezymey1.pro',
 'bintradbasina.com',
 'bit3-1lexipro.com',
 'bitcash.cash',
 'bitcoincashbch.com',
 'bitfarm.cc',
 'bitfxcrypto.com',
 'bitprofits.net',
 'bll6pcr2.sa.com',
 'bonusmaniac.lotteryjackpot.net',
 'braxfiace.com',
 'brightia.digital',
 'brokerstrategy.top',
 'bron-invionsystem.com',
 'btc-850-epeak.com',
 'btgfxtrade.com',
 'buildstock-finance.com',
 'bullexptrades.com',
 'byex.info',
 'campwrk.top',
 'cannafo.com',
 'canvasguardpro.digital',
 'capitalfundlab.com',
 'capitaltrademiners.com',
 'capraiserstrd.com',
 'captcha-kra30-cc.ru',
 'cartera.capital',
 'cascade-nexus.com',
 'cfd.uniglobal-group.io',
 'chiaroinvexus-app.com',
 'chiken-farm-original.org',
 'chivotrade.com',
 'cinema-hd.tv',
 'clarodexeris.net',
 'clubcelestestar.com',
 'coinwaden.com',
 'cracksofshah.com',
 'creditstrategy.top',
 'crowncryptotrading.com',
 'cryptokenassets.com',
 'cryptoliferadings.live',
 'cryptotrademasters.com',
 'csd-bfl.ru',
 'cvzxpnew-vtb24gidr.ru',
 'cyriptosglobal-fx.com',
 'd8.ebook.surf',
 'dacsincstore.com',
 'darknetlive.cc',
 'dcura.ru',
 'defiee.com',
 'detisakhalina.ru',
 'dexartrades.org',
 'digitalcorefex.com',
 'digitalmarketingsecrets.sa.com',
 'dlyhx.com',
 'dsx.obmenvsemfiles.net',
 'dveri-s-termorazryvom.ru',
 'edydlploms.com',
 'elitetradehive.com',
 'elvengold.net',
 'emeraldtrustltd.com',
 'emovobi29.decoration-fleuriste-lyon.fr',
 'empresafinanceiragalf.com',
 'endophin.ru',
 'energybikers.ru',
 'equinoversefinance.com',
 'equiti-global-market.com',
 'eternaserialwhr.online',
 'ethereumxphiprexsolution.com',
 'euromedex.ru',
 'exantcorpdevelopfx.com',
 'excellent-capitals.com',
 'exceptionaltradex.com',
 'expert-traders.website',
 'exponentskate.com',
 'expressauracargo.com',
 'faberlic-catalog-online.ru',
 'familyhealthylifestyle.sa.com',
 'famousfinance.net',
 'fastflowvaltrix-soft.com',
 'faultless-security.ru',
 'fbbc.pro',
 'ferrum.ro',
 'fidelefintrion.com',
 'filmfeed.online',
 'financelegendai.com',
 'finmaxbo.com',
 'finpros-exchange.com',
 'fiscalfomulasfx.com',
 'flerturexapp.net',
 'flexcapitaltrades.com',
 'fluxorbeamai.com',
 'fluxxcapital.site',
 'fly-traders.com',
 'fonterranks.com',
 'forexbrokagepro.com',
 'forextrade.ltd',
 'ftfinances.com',
 'ftse100.org',
 'funds-funding.digital',
 'futurepfs.com',
 'fx-official.org',
 'fxcmmtrade.com',
 'fxcmtraderfx.com',
 'fxdurationmining.com',
 'fxminerstrade.com',
 'fxproru.group',
 'fxprosupatrade.net',
 'fxtm-uk.net',
 'g-easytdg.com',
 'g10y.top',
 'g13q.top',
 'g16z.top',
 'g18f.top',
 'g19m.top',
 'g20h.top',
 'g23e.top',
 'g27x.top',
 'galvonixtrader.net',
 'gatecapitals.com',
 'gay-baza.bond',
 'gblksfx.com',
 'geminiwealthwinnings.com',
 'getx-app.com',
 'gitu.countaddflip.cfd',
 'gleem-capex.com',
 'globalcredit.cc',
 'gomakete.com',
 'good-pay.eu',
 'googletune.com',
 'grandbestheritageglobal.com',
 'grandforexworldwide.com',
 'gsefx.com',
 'guapvest.live',
 'gzepk.com',
 'hantecinvestments.com',
 'happypetcaretips.sa.com',
 'harborviewtrust.com',
 'hashora.top',
 'helpnds.ru',
 'highlvltrades.com',
 'hindcine.tv',
 'horizonfinancetrade.com',
 'horizontrading.live',
 'https-kra34--cc.ru',
 'hydromod.online',
 'hypenfinancegalf.com',
 'i-fast-cptl-casn.net',
 'ibkmarkets.online',
 'ifxbestrade.com',
 'ifxtrade.club',
 'ifxwowinvest.org',
 'immediate-achieve.net',
 'immediateaibank.com',
 'immediatefastx.net',
 'insightmarketplatform.com',
 'internationalfxtrendsignal.com',
 'internationalfxtrendsignals.com',
 'intrastock.click',
 'investacorps.com',
 'investobruchev.ru',
 'invexgpt-v9-engine.com',
 'irt-auto.ru',
 'irwin2397.buzz',
 'irwin3210.buzz',
 'irwin8320.buzz',
 'irwin8787.buzz',
 'ivyace.com',
 'joypath.shop',
 'jrlendib.com',
 'jugsool.ru',
 'kakdoma54.ru',
 'kakhranitedy.ru',
 'kakoy-chelovek.ru',
 'karaokestmarks.com',
 'karkas-dom-stroy.ru',
 'kazino-gouaild.vtlaw1.com',
 'keymanagementinsights.com',
 'kingaszczesliwa.pl',
 'kinogo.oneproxy.xyz',
 'kinotik.org',
 'kiwap.cc',
 'kkraaa38cc.ru',
 'koalwholesale.com',
 'kolov.info',
 'komfortno24.ru',
 'koronapay.com-uni.asia',
 'koronapay.com-web.asia',
 'koronapay.loadalink.buzz',
 'koronapay.receivei.sbs',
 'kra-----37--------cc.ru',
 'kra----36---------cc.ru',
 'kra----36at.ru',
 'kra----37------cc.ru',
 'kra-5at.com',
 'kra-aa-31-at.ru',
 'kra142.cc',
 'kra37portal.ru',
 'kra38.bet',
 'krak4-at.com',
 'kraken-ai.net',
 'kraken33.net',
 'kraken4you.com',
 'krakendarknetor.net',
 'krakensite.net',
 'krakenvk.com',
 'krakssylka.cc',
 'krareg.cc',
 'kudixhood.com',
 'la-atmosfera.com',
 'legacywealthtrades.com',
 'legiohliberty.army',
 'legionliberty.cc',
 'legionlibertys.army',
 'legionlibertys.cc',
 'leon-0up9.buzz',
 'leon-bet.net',
 'libradonetsk.ru',
 'lite-finance-group.com',
 'litemin-hubs.com',
 'livegrowthinvestments.com',
 'loveshope13.biz',
 'loysci.com',
 'lumiar-bitrow.com',
 'm3ga.ee',
 'madeinslavutych.org',
 'maestropc.com',
 'magazinapelsin.ru',
 'magic-lime.info',
 'marathondigitaltips.com',
 'marcelonevesurologia.com.br',
 'matrix-world.info',
 'maximarkets.mobi',
 'maxqalcg.info',
 'maxstrides.com',
 'mebel-nizhniy.ru',
 'mebelvcamare.ru',
 'mebelvsamaru.ru',
 'mecury-coins.com',
 'mega-sb.biz',
 'megawe16.at',
 'mentorshipinvestment.com',
 'mfb.changeai.top',
 'mgmarkett6.at',
 'minecraftors.ru',
 'mirante-fund-management.com',
 'mkrn.cc',
 'mmkinvestment.com',
 'money-fund.net',
 'moranoin.com',
 'morganonemarkets.com',
 'moscowstudentlife.ru',
 'mostbet-in46.com',
 'mostbet-wnz7.top',
 'mtg-invest.com',
 'multi-royalq-trading.net',
 'muvipoisk.net',
 'myearn.changeai.top',
 'myfarm.redheadcode.com',
 'mylooppassivefc.com',
 'nanufyo1.pro',
 'nedvc.com',
 'netto-paytorn-system.com',
 'new-luk.firebaseapp.com',
 'newsbuzzing.ru',
 'nextquix.com',
 'nextwa-ve.com',
 'nexusgen-limited.com',
 'nice-wear.ru',
 'nixubay5.pro',
 'nostrospro.com',
 'nvu-bankrot-netdolgov.ru',
 'octatradeglobal.com',
 'official-ru.npbfx.nl',
 'official.npbfx.nl',
 'officialru.npbfx.nl',
 'omegablockitex.net',
 'onion-kraken.net',
 'optimalsolutionhub.com',
 'optimumgoldshares.com',
 'opulatrix-app.net',
 'opulentassetsonline.com',
 'oraqsmart.online',
 'oraqsmart.org',
 'otkrytdah.com',
 'otvetimfaq.ru',
 'paycoin.store',
 'paymorebit.com',
 'pc.charlesent.com',
 'pc.cornzormus.vip',
 'pc.fxcmmeultd.com',
 'pc.fxgtlife.cc',
 'pc.gblksfx.cc',
 'pc.iwaicosmo.cc',
 'pc.mtofxus.cc',
 'pc.nebxuw.com',
 'peretti-bitvalor-engine.com',
 'peturo.com',
 'pg-yieh0eegae.global.e-cloud.ch',
 'phantomexperts.com',
 'picsmeplease.com',
 'piratez.xyz',
 'platform-kraken.cc',
 'pliage.ru',
 'plotntk.ru',
 'podyaka.org',
 'port.hair',
 'potomu4toiabatman.ru',
 'poverka23.ru',
 'pp-nata-fit.ru',
 'ppatio.ru',
 'ppcglobal.online',
 'prefimasset.com',
 'premium-trade.net',
 'primsphermarkting.com',
 'privedite-primery.ru',
 'produktioptom.ru',
 'proearners.live',
 'profattestatia.ru',
 'profitablenes.org',
 'profitmasstrades.net',
 'profitpine.com',
 'protracker.fun',
 'ps-eestienergia4-ru.web.app',
 'pskimgs.cc',
 'putlockerc.to',
 'pvfinvestment.com',
 'qrec3.hivor.top',
 'qu-antity.vip',
 'quacauvong.com',
 'ravto-vault-ai.com',
 'rdk-contact.com',
 'rdk-official.com',
 'rdk.social',
 'rdkcontact.com',
 'refpaeovcmka.top',
 'registergasoffer.com',
 'reinkarnaciya-bezrabotnogo.ru',
 'rem-tv-spb.ru',
 'rennerph.ru',
 'rituall-vechnost.ru',
 'robbin-hood.org',
 'robinhoodinvestmentsplatform.com',
 'rojadirectatv.club',
 'rosecapital.ltd',
 'rovenmill-app.com',
 'rufuswainwrightvip.com',
 'rusudo-xxjodgg-xxkghnn.top',
 'rusvokcorps.com',
 'rusvolcorp.com',
 'rusvolcorps.net',
 'ruvolcorps.com',
 'rzwab.com',
 'samsung-galaxy.mobi',
 'san-dana.ru',
 'santehnikanf.ru',
 'sarkoidoza.ru',
 'sartradingmarket.com',
 'sbotopkr.com',
 'schiff-vaults.pro',
 'schrodersinvestment.net',
 'selektorkazino.cloud',
 'serpcarparts.ru',
 'shpok.top',
 'silentbet.lotteryjackpot.net',
 'skolkovariantov.ru',
 'skyecryptofx.com',
 'smartminingusdt.mom',
 'smbpipsmaster.com',
 'sol-bet.com',
 'souzfs.ru',
 'sovmestimost-info.ru',
 'stakeallcoins.com',
 'stakesite.click',
 'staralien.ru',
 'starlitfxtrade.com',
 'stocksminermarket.com',
 'strana.phic4.top',
 'stressbalans.com',
 'studio54.biz',
 'stylishhomeinteriorconcepts.sa.com',
 'surprisingprojectt.shop',
 'swapadipex.com',
 'sway-fxtrade.live',
 'swift-vionexsolution.com',
 'swiftoption.live',
 'syjzt.com',
 'synalea5.pro',
 'tamsignalsview.com',
 'taxi-alina.ru',
 'tbstradingstock.com',
 'tcons.ru',
 'techgeekss.com',
 'technogas40.ru',
 'tervin-axorium-app.com',
 'teslafastminning.com',
 'thewealthhubs.com',
 'tigr-inbkers.com',
 'tischlerei-hoff.de',
 'tntx.cc',
 'topassetholdings.com',
 'topmyz.com',
 'tracefxpaypro.click',
 'trackadipex-neo.com',
 'trackwebsync.xyz',
 'tradahive.com',
 'tradahivee.com',
 'tradahivve.com',
 'trade-8flarexbot-com.cryptofinancetrack.com',
 'trade-spark.live',
 'tradecopypro.net',
 'tradeinjectfox.com',
 'tradepromarket.net',
 'traderinvestmentt.com',
 'traderxmining.org',
 'traffic-dxn.com',
 'traffic-fft.com',
 'trickotage.ru',
 'triloxai-app.com',
 'trust-equityfx.com',
 'trustfndsfxpro.online',
 'tsinvester.com',
 'turfcapprivate.com',
 'turiwhan.top',
 'tyrioncapital.pro',
 'ulgran73.ru',
 'ultrcorp.com',
 'unionunlimitedfx.com',
 'unlimitedfxs.com',
 'untsug.mn',
 'us-atrade.com',
 'usdtventures.com',
 'v-g.cc',
 'v-teme.com',
 'vaultedgeinvestments.com',
 'vaultorox.com',
 'vavada-3wse.buzz',
 'vavada-7108.buzz',
 'vavada20001.com',
 'vavada999e.com',
 'vavadabb27.com',
 'vavadafedc.com',
 'velikolepnyi-vek.ru',
 'velminoption.click',
 'verhclub.ru',
 'videopas.info',
 'vigorcrestflow.com',
 'vip-mood.quest',
 'vipsgt.com',
 'vital-xbc.ru',
 'vitolalpha.top',
 'vitrademarket.com',
 'vladimirkazadaev.ru',
 'vlchz.ru',
 'vlcne777.click',
 'voprosy-migranta.ru',
 'vsyoproogorod.ru',
 'vtb24-holinburg.online',
 'walkmarketsystem.com',
 'webfox.ca',
 'welcome100.npbpromoru.nl',
 'winning-eldi.xyz',
 'wisdomweb.xyz',
 'worldcryptotrades.com',
 'wowmodels.ru',
 'xanal.xyz',
 'xchangetd.com',
 'xn--kr29-1na.cc',
 'xnorai.net',
 'xntiaoji.com',
 'xtb-onlinetrading.com',
 'xtbpro.com',
 'xunijuo5.pro',
 'yookassa.studiophoto.top',
 'zastrijkoy.ru',
 'zellatradeinvesment.com',
 'zenithqueststrategists.com',
 'zhivayastal.ru',
 'zksyncdigitalmining.com',
 'znacheniya-imyon.ru',
 'zynserax-system-com.cryptofinancetrack.com'
}

In [55]:
if SAFE_BROWSING_KEY and os.environ.get("USE_GOOGLE_SAFE_BROWSING_KEY"):
    flagged = safe_browsing_flags(rkn_filtered, SAFE_BROWSING_KEY)
else:
    flagged = manual_safe_browsing_run
    print("ключ не задан, применяю сайты из ручного прогона")

rkn_filtered = [d for d in rkn_filtered if d not in flagged]
print(f"отсеяно Safe Browsing: {len(flagged):,}")
print(f"итог ru-blocked: {len(rkn_filtered):,}")

отсеяно Safe Browsing: 540
итог ru-blocked: 519,979


## 7. Сборка geosite.dat

Файл пишется напрямую в protobuf. Официальный компилятор `domain-list-community`
требует Go ≥ 1.25, которого нет в пакетах Ubuntu — установка тулчейна в CI ради
сериализации трёх вложенных сообщений не оправдана.

Схема из `app/router/config.proto` v2ray-core:

```
message Domain {
  enum Type { Plain = 0; Regex = 1; Domain = 2; Full = 3; }
  Type   type  = 1;
  string value = 2;
}
message GeoSite     { string country_code = 1; repeated Domain domain = 2; }
message GeoSiteList { repeated GeoSite entry = 1; }
```

In [56]:
TYPE_DOMAIN = 2  # правило с покрытием поддоменов


def _varint(value):
    out = bytearray()
    while True:
        byte = value & 0x7F
        value >>= 7
        out.append(byte | 0x80 if value else byte)
        if not value:
            return bytes(out)


def _tag(field, wire):
    return _varint((field << 3) | wire)


def _len_delim(field, payload):
    return _tag(field, 2) + _varint(len(payload)) + payload

In [57]:
def encode_geosite(lists):
    """{имя списка: [домен, ...]} → содержимое geosite.dat.

    Имена приводятся к верхнему регистру: так они хранятся во всех
    публикуемых .dat, обращение geosite:ru-blocked регистронезависимо.
    """
    out = bytearray()
    for name, domains in lists.items():
        entry = _len_delim(1, name.upper().encode())
        for domain in domains:
            rule = _tag(1, 0) + _varint(TYPE_DOMAIN) + _len_delim(2, domain.encode())
            entry += _len_delim(2, rule)
        out += _len_delim(1, entry)
    return bytes(out)

In [58]:
def _read_varint(buf, pos):
    result = shift = 0
    while True:
        byte = buf[pos]
        pos += 1
        result |= (byte & 0x7F) << shift
        if not byte & 0x80:
            return result, pos
        shift += 7


def _iter_fields(buf):
    pos = 0
    while pos < len(buf):
        key, pos = _read_varint(buf, pos)
        field, wire = key >> 3, key & 7
        if wire == 0:
            value, pos = _read_varint(buf, pos)
            yield field, value
        elif wire == 2:
            length, pos = _read_varint(buf, pos)
            yield field, buf[pos:pos + length]
            pos += length
        else:
            raise ValueError(f"неподдерживаемый wire type {wire}")

In [59]:
def decode_geosite(raw):
    """Обратный разбор — используется для проверки записанного файла."""
    result = {}
    for field, payload in _iter_fields(raw):
        if field != 1:
            continue
        name, domains = None, []
        for sub_field, sub_payload in _iter_fields(payload):
            if sub_field == 1:
                name = sub_payload.decode()
            elif sub_field == 2:
                value = None
                for dom_field, dom_payload in _iter_fields(sub_payload):
                    if dom_field == 2:
                        value = dom_payload.decode()
                if value:
                    domains.append(value)
        if name:
            result[name] = domains
    return result

In [60]:
# Домен, попавший в обе ветки, остаётся только в ru-forbidden: маршрут для
# обеих один, дублирование расходует память клиента.
ru_forbidden = dedup_by_coverage(geoblock_clean)
forbidden_set = set(ru_forbidden)
ru_blocked = [d for d in dedup_by_coverage(rkn_filtered) if d not in forbidden_set]

print(f"ru-blocked:   {len(ru_blocked):,}")
print(f"ru-forbidden: {len(ru_forbidden):,}")

ru-blocked:   519,812
ru-forbidden: 1,140


In [61]:
os.makedirs("publish", exist_ok=True)

with open("publish/geosite.dat", "wb") as f:
    f.write(encode_geosite({"ru-blocked": ru_blocked, "ru-forbidden": ru_forbidden}))

for name, domains in (("ru-blocked", ru_blocked), ("ru-forbidden", ru_forbidden)):
    with open(f"publish/{name}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(domains) + "\n")

print(os.listdir("publish"))

['ru-blocked.txt', 'geosite.dat', 'ru-forbidden.txt']


In [62]:
raw = open("publish/geosite.dat", "rb").read()
parsed = decode_geosite(raw)

assert set(parsed) == {"RU-BLOCKED", "RU-FORBIDDEN"}
assert parsed["RU-BLOCKED"] == ru_blocked
assert parsed["RU-FORBIDDEN"] == ru_forbidden
print("проверка разбором пройдена:", {k: len(v) for k, v in parsed.items()})

проверка разбором пройдена: {'RU-BLOCKED': 519812, 'RU-FORBIDDEN': 1140}


In [63]:
size_mb = len(raw) / 1024 / 1024
print(f"размер geosite.dat: {size_mb:.2f} МБ")
if size_mb > 25:
    print("превышает бюджет 25-35 МБ для NEPacketTunnelProvider:")
    print("  оставить в рабочем наборе меньше категорий или удалять корзину")
    print("  просмотра (rkn_filtered = kept)")

размер geosite.dat: 11.06 МБ
